[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Beanie Documents


## What you will be able to do

Write a class that is a Pydantic model and a MongoDB collection at once, connect it with
`init_beanie`, and insert and read documents as typed objects rather than dictionaries. Use `await`
at the top level of a notebook cell, which is the first time this guide has needed it. Say what
`init_beanie` does and what every one of its failures means, including the one raised by a document
that predates the model. And recognize the missing `await`, which here leaves you holding a query
object that is perfectly truthy.


## The idea

### The problem

Everything so far has been dictionaries. A dictionary has no shape, so nothing checks that `price`
is a number, nothing stops a typo creating a new field, and every read hands you `document["name"]`
with no idea whether that key exists.

Beanie puts a Pydantic model in front of the collection. The model is the schema, and it is checked
in Python before anything is sent.

### What a Document is

A subclass of `beanie.Document`, which is a Pydantic `BaseModel` with an `id` field and methods that
talk to a collection. The class is the collection; instances are documents.

### Why init_beanie exists

The class is defined before there is a database, so something has to connect the two. `init_beanie`
takes a database and a list of models, binds each to its collection, and builds any indexes the
models declare. Until it has run, the model knows nothing and every query raises.

### Where this shows up

Every Beanie program starts with it, usually in a startup handler, and every Beanie program's first
error is either that it was not called or that the connection string had no database name on it.

### What this notebook covers

Defining a `Document` and its `Settings`. `init_beanie`. Inserting, finding and deleting as typed
objects. `await` in a notebook, and the two event loop errors around it. Then the four failures,
including the validation error for data written before the model existed.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

from beanie import Document, init_beanie
from pymongo import AsyncMongoClient


class Product(Document):                           # a Pydantic model that is also a collection
    sku: str
    name: str
    price: float

    class Settings:
        name = "catalog"


async def main():
    client = AsyncMongoClient("mongodb://127.0.0.1:27017/shop")
    await init_beanie(database=client.get_default_database(), document_models=[Product])

    await Product.delete_all()
    await Product(sku="LAP-1", name="a laptop", price=999.0).insert()

    found = await Product.find_one(Product.sku == "LAP-1")
    print("read back as a model:", found.name, "| price plus one:", found.price + 1)
    print("with an id from the database:", type(found.id).__name__)

    try:
        Product(sku="X", name="x", price="free")
    except Exception as error:
        print("and a bad price never reaches MongoDB:", type(error).__name__)

    await client.close()


asyncio.run(main())
```

```
read back as a model: a laptop | price plus one: 1000.0
with an id from the database: PydanticObjectId
and a bad price never reaches MongoDB: ValidationError
```

`found.name` rather than `found["name"]`, and `found.price + 1` works because `price` really is a
float. The last line is the part that was not available before: a bad value is refused in Python,
by the model, before a byte is sent.


## Setup

Twelve imports, MongoDB, the boot cell, and two helpers.

- `beanie` with `Document` and `init_beanie`, and `pydantic` so the version can be printed
- `pymongo` with `AsyncMongoClient`, which is what Beanie 2 runs on. **Motor is gone**: it was the
  asynchronous driver Beanie used until PyMongo grew its own, and you will still find it in older
  tutorials and in Beanie 1's documentation
- `IndexModel` declares an index on a model, `asyncio` is for one error below
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

`first_problem` pulls one line out of a Pydantic `ValidationError`, which otherwise prints several
paragraphs and a documentation link. `failed` does the same for a MongoDB failure.


In [1]:
import asyncio
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pydantic
import pymongo
from beanie import Document, init_beanie
from pymongo import AsyncMongoClient, IndexModel

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def first_problem(error):
    """One line out of a pydantic ValidationError, which is otherwise several paragraphs."""
    problem = error.errors()[0]
    return f"{'.'.join(str(part) for part in problem['loc'])}: {problem['msg']}"


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())
print("beanie", beanie.__version__, "| pydantic", pydantic.__version__)


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500
beanie 2.2.0 | pydantic 2.13.5


## Worked examples

### A model that is a collection

Note the `await` with no `asyncio.run` around it. A notebook cell is already inside an event loop,
which is the whole reason this guide could not start with Beanie:


In [2]:
class Product(Document):
    sku: str
    name: str
    price: float
    stock: int = 0                                                  # a default, as in any model

    class Settings:
        name = "catalog"                                            # the collection it lives in


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Product])

print("bound to collection:", Product.get_pymongo_collection().name)
print("fields:", list(Product.model_fields))


bound to collection: catalog
fields: ['id', 'revision_id', 'sku', 'name', 'price', 'stock']


`id` is there without being declared: `Document` adds it, and it is `None` until the document has
been inserted. `Settings.name` is the collection; without it Beanie uses the lower-cased class name.

### Writing and reading


In [3]:
await Product.delete_all()

laptop = Product(sku="LAP-1", name="a laptop", price=999.0, stock=3)
print("before insert, id is:", laptop.id)

await laptop.insert()
print("after insert, id is a:", type(laptop.id).__name__)

found = await Product.find_one(Product.sku == "LAP-1")
print("found:", found.name, "| stock:", found.stock, "| a real float:", found.price / 3)


before insert, id is: None
after insert, id is a: PydanticObjectId
found: a laptop | stock: 3 | a real float: 333.0


`Product.sku == "LAP-1"` is not a comparison. It builds a query, which is why it can be passed to
`find_one`, and it is checked against the model: a mistyped field name is an `AttributeError`
immediately rather than a filter that matches nothing.

### Several at once, and the query API


In [4]:
await Product.delete_all()
await Product.insert_many([
    Product(sku="LAP-2", name="another laptop", price=1200.0, stock=1),
    Product(sku="MON-1", name="a monitor", price=300.0, stock=8),
    Product(sku="CAB-1", name="a cable", price=9.0, stock=100),
])

everything = await Product.find_all().to_list()
print("all:", [product.sku for product in everything])

cheap = await Product.find(Product.price < 500).to_list()
print("under 500:", [product.sku for product in cheap])

print("count:", await Product.find(Product.stock > 0).count())


all: ['LAP-2', 'MON-1', 'CAB-1']
under 500: ['MON-1', 'CAB-1']
count: 3


`find_all()` and `find(...)` return a query object, and `to_list()` is what runs it. The query object
is lazy in the same way a cursor is, which matters for the error below.

### The typo that is caught, and the one that is not


In [5]:
try:
    await Product.find(Product.prce < 500).to_list()                # a mistyped field
except AttributeError as error:
    print("through the model:", type(error).__name__ + ":", error)

silent = await Product.find({"prce": {"$lt": 500}}).to_list()       # the same typo, as a dict
print("as a raw dictionary: ", len(silent), "documents, and no error")


through the model: AttributeError: prce
as a raw dictionary:  0 documents, and no error


That is the argument for the expression form in one cell. `Product.prce` does not exist, so Python
says so. `{"prce": ...}` is a perfectly good filter for a field no document has, so MongoDB answers
honestly with nothing.

Beanie accepts both, and the raw dictionary is there for the queries the expression form cannot
build. Prefer the expressions.

### What the model refuses


In [6]:
for description, fields in (("a price that is not a number", {"sku": "X", "name": "x",
                                                              "price": "free"}),
                            ("a missing field", {"sku": "X", "name": "x"}),
                            ("stock as a float", {"sku": "X", "name": "x", "price": 1.0,
                                                  "stock": 1.5})):
    try:
        Product(**fields)
        print(f"  {description:29} accepted")
    except pydantic.ValidationError as error:
        print(f"  {description:29} {first_problem(error)}")


  a price that is not a number  price: Input should be a valid number, unable to parse string as a number
  a missing field               price: Field required
  stock as a float              stock: Input should be a valid integer, got a number with a fractional part


None of those reached MongoDB. The third is worth a look: Pydantic refuses `1.5` for an `int` rather
than truncating it, which is the behavior you want and is not what a plain `dict` would have done.

Note also that `"999"` as a price **would** be accepted and converted, because Pydantic coerces a
numeric string to a float. The model is a contract about types, not about formats.

### Indexes on the model


In [7]:
class Indexed(Document):
    sku: str
    name: str

    class Settings:
        name = "indexed_catalog"
        indexes = [IndexModel([("sku", 1)], name="sku_unique", unique=True)]


await client.get_default_database().indexed_catalog.drop()
await init_beanie(database=client.get_default_database(), document_models=[Indexed])

print("indexes built by init_beanie:",
      [index["name"] async for index in await Indexed.get_pymongo_collection().list_indexes()])


indexes built by init_beanie: ['_id_', 'sku_unique']


`init_beanie` builds them, every time it runs, which is safe because creating an index that already
exists does nothing. It is also why a model whose declared index disagrees with one already in the
database fails at startup rather than at the first query.

### When to reach for which

| What you want | How to write it |
|---|---|
| a collection | a class subclassing `Document` |
| its name | `class Settings: name = "..."` |
| to connect | `await init_beanie(database=..., document_models=[...])` |
| one document | `await Model.find_one(Model.field == value)` |
| several | `await Model.find(...).to_list()` |
| all of them | `await Model.find_all().to_list()` |
| to write one | `await instance.insert()` |
| to write many | `await Model.insert_many([...])` |
| a count | `await Model.find(...).count()` |
| an index | `class Settings: indexes = [IndexModel(...)]` |
| a query the expressions cannot build | a raw dictionary, knowingly |

The default is the expression form, because it is checked against the model. Reach for a raw
dictionary only when you need an operator Beanie has no expression for, and expect no help from
Python when you do.

### A catalog with a model in front of it, finished


In [8]:
class Catalogued(Document):
    sku: str
    name: str
    price: float
    stock: int = 0

    class Settings:
        name = "catalogued"
        indexes = [IndexModel([("sku", 1)], name="sku_unique", unique=True)]

    def is_available(self):
        """A method on the document, which a dictionary could never have."""
        return self.stock > 0


async def stock_up(rows):
    """Validate every row before any of it is written, then write it in one call."""
    products = [Catalogued(**row) for row in rows]                  # raises here if a row is bad
    await Catalogued.insert_many(products)
    return len(products)


await client.get_default_database().catalogued.drop()
await init_beanie(database=client.get_default_database(), document_models=[Catalogued])

print("written:", await stock_up([
    {"sku": "A-1", "name": "a thing", "price": 10.0, "stock": 2},
    {"sku": "A-2", "name": "another", "price": 20.0},
]))

for product in await Catalogued.find_all().sort(Catalogued.sku).to_list():
    print(f"  {product.sku}  {product.name:9} {product.price:7.2f}  available:",
          product.is_available())

try:
    await stock_up([{"sku": "A-3", "name": "fine", "price": 1.0},
                    {"sku": "A-4", "name": "bad", "price": "free"}])
except pydantic.ValidationError as error:
    print("a bad row stopped the whole batch:", first_problem(error))
print("still", await Catalogued.find_all().count(), "documents")


written: 2
  A-1  a thing     10.00  available: True
  A-2  another     20.00  available: False
a bad row stopped the whole batch: price: Input should be a valid number, unable to parse string as a number
still 2 documents


The important line is the one that builds the list before writing anything. Because validation
happens when the objects are constructed, a bad row raises before `insert_many` is called, so the
batch is all or nothing without a transaction.

That is a property of putting the model in front, and it is the opposite of the half-finished
`insert_many` in **Bulk Writes and Transactions**, where the check was on the server and the first
half had already landed.

### Where each part came from

| In the catalog | What it relies on | The section that showed it |
|---|---|---|
| `class Catalogued(Document)` | a model that is also a collection | A model that is a collection |
| `Settings.indexes` | `init_beanie` building them | Indexes on the model |
| `Catalogued(**row)` raising | validation at construction | What the model refuses |
| `insert_many` after the loop | every row checked before any is sent | What the model refuses |
| `is_available` | a method on the document | A model that is a collection |
| a unique `sku` | a constraint that is also an index | **Indexes** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/11-beanie-documents-solutions.ipynb).

**1.** Define a `Document` with three fields and connect it with `init_beanie`.


In [9]:
# your code here


**2.** Insert one and print the `id` before and after.


In [10]:
# your code here


**3.** Insert three and find the ones matching a condition.


In [11]:
# your code here


**4.** Show that a mistyped field raises through the model and matches nothing as a dictionary.


In [12]:
# your code here


**5.** Try to build a document with a bad value and print one line of the error.


In [13]:
# your code here


**6.** Declare a unique index on a model and show `init_beanie` built it.


In [14]:
# your code here


## Common errors

### RuntimeError: asyncio.run() cannot be called from a running event loop


In [15]:
async def count_them():
    return await Product.find_all().count()


asyncio.run(count_them())


RuntimeError: asyncio.run() cannot be called from a running event loop

Every Beanie example you will find begins with `asyncio.run(main())`, because most of them are
scripts. A notebook cell is already running inside an event loop, and a loop cannot be started
inside a loop.

Drop the wrapper. `await` works at the top level of a cell:


In [16]:
print("just await it:", await count_them())


just await it: 3


There is a library that patches the loop so the nesting works, and it is worth saying plainly that
you do not need it: in a notebook `await` is the supported way, and in a script `asyncio.run` is.

### beanie.exceptions.CollectionWasNotInitialized


In [17]:
class Unbound(Document):
    name: str


try:
    await Unbound.find_all().to_list()
except beanie.exceptions.CollectionWasNotInitialized as error:
    print(f"{type(error).__module__}.{type(error).__name__}, with an empty message: {str(error)!r}")


beanie.exceptions.CollectionWasNotInitialized, with an empty message: ''


The class exists, the query builds, and there is no collection behind it because `init_beanie` was
never told about this model. The message is empty, which makes the class name the only information
you get.

Every model must be in the `document_models` list:


In [18]:
await init_beanie(database=client.get_default_database(),
                  document_models=[Product, Unbound])
print("now it works:", await Unbound.find_all().to_list())


now it works: []


### pymongo.errors.ConfigurationError: No default database name defined or provided


In [19]:
no_database = AsyncMongoClient("mongodb://127.0.0.1:27017")         # no /shop on the end
try:
    await init_beanie(database=no_database.get_default_database(), document_models=[Product])
except pymongo.errors.ConfigurationError as error:
    print(f"{type(error).__module__}.{type(error).__name__}: {error}")
await no_database.close()


pymongo.errors.ConfigurationError: No default database name defined or provided.


`get_default_database()` reads the database name off the end of the connection string, and there is
none. This is why every URI in this guide is `mongodb://127.0.0.1:27017/shop` rather than stopping at
the port.

Either put it in the URI or name it explicitly, and the second is clearer when the code is read
later:


In [20]:
explicit = AsyncMongoClient("mongodb://127.0.0.1:27017")
await init_beanie(database=explicit["shop"], document_models=[Product])
print("named explicitly:", await Product.find_all().count(), "products")

await explicit.close()
await init_beanie(database=client.get_default_database(),   # bind back to the open client, or every
                  document_models=[Product, Unbound])       # later cell uses one that is closed


named explicitly: 3 products


### pydantic_core.ValidationError: the documents that predate the model


In [21]:
raw = client.get_default_database().catalog
await raw.insert_one({"sku": "OLD-1", "name": "written before price existed"})

try:
    await Product.find_all().to_list()
except pydantic.ValidationError as error:
    print("reading the collection now fails:", first_problem(error))

await raw.delete_many({"sku": "OLD-1"})
print("and after removing it:", await Product.find_all().count(), "products")


reading the collection now fails: price: Field required
and after removing it: 3 products


The collection has no schema, so a document written by anything else, or by an earlier version of
this model, sits there quite happily until a Beanie query reads it and Pydantic refuses it.

This is the cost of validation in the client: nothing stopped the write, so the check lands on the
reader. Two ways out, and they are different jobs. A default makes the model tolerant of what is
already there:


In [22]:
class Tolerant(Document):
    sku: str
    name: str
    price: float = 0.0                                              # a default for old documents

    class Settings:
        name = "catalog"


await init_beanie(database=client.get_default_database(), document_models=[Tolerant])
await raw.insert_one({"sku": "OLD-2", "name": "still no price"})

for product in await Tolerant.find(Tolerant.sku == "OLD-2").to_list():
    print("read with a default:", product.sku, "| price:", product.price)

removed = await raw.delete_many({"sku": "OLD-2"})                   # named, so the cell prints nothing
print("cleaned up:", removed.deleted_count)


read with a default: OLD-2 | price: 0.0
cleaned up: 1


The other way is to make the database enforce the shape too, with a `$jsonSchema` validator on the
collection, so that nothing can write a document the model would refuse. Beanie does not add one for
you, and **Migrations** is where changing the documents you already have belongs.


In [23]:
await client.get_default_database().catalog.drop()
await client.get_default_database().catalogued.drop()
await client.get_default_database().indexed_catalog.drop()
await client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- A `Document` subclass is a Pydantic model and a collection at once. `Settings.name` names the
  collection and `Settings.indexes` declares indexes `init_beanie` will build.
- `id` is added for you and is `None` until the document is inserted.
- `await init_beanie(database=..., document_models=[...])` binds models to collections. Until it
  runs, every query raises `CollectionWasNotInitialized`, whose message is empty.
- `get_default_database()` needs a database name on the connection string, or
  `ConfigurationError`. Naming it as `client["shop"]` is clearer anyway.
- A notebook cell is already in an event loop, so `await` works at the top level and `asyncio.run`
  raises. A script is the other way round.
- `Model.field == value` builds a query checked against the model, so a typo is an `AttributeError`.
  The same typo in a raw dictionary matches nothing and raises nothing.
- Validation happens when the object is constructed, so a bad row in a batch stops the batch before
  anything is written.
- A document written before the model existed raises `ValidationError` when read. A field default
  makes the model tolerant; a `$jsonSchema` validator stops the writes at the database.
- Beanie 2 runs on PyMongo's `AsyncMongoClient`. Motor is gone.


## What is next

**Async Queries** is the query API in full: `find` with several conditions, the operators module,
sorting and limiting, and the projection model whose field names have to match what the pipeline
actually produced.


---

&#8592; **Previous:** [Modeling Without Joins](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/10-modeling-without-joins.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Async Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/12-async-queries.ipynb) &#8594;
